In [ ]:

import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

N=36
t_months = np.arange(N)
true_trend = 0.5*t_months # steadily increasing (no exponential type change)
# the pymc model for expo trend would be diff than what we do now

# quantity per month type, the slope

true_seasonality = 10 * np.sin(2 * np.pi * t_months / 12)
# sine for the cyclic shock # Asint, sine wave shm

demand_total = true_trend + true_seasonality + np.random.normal(0, 2, N)

plt.plot(t_months, demand_total, "k.", label="Raw Chaos (Data)")
plt.plot(t_months, true_trend + true_seasonality, "r-", label="True Signal (Drift + Monsoon)")
plt.legend()
plt.title("Forensic Baseline: The Underlying Physical Reality")
plt.show()

# bayesian structual time series (seasonality)

# split the time series vairables into explicit blocks
# (sine cosine fourier pairs)

In [ ]:
with pm.Model() as bsts_full:

    # Structural Priors
    sigma_trend = pm.HalfNormal("sigma_trend", sigma=2)
    drift = pm.Normal("drift", mu=0.5, sigma=1) # since our reality is an upward trend
    trend_component = pm.GaussianRandomWalk("trend_component", mu=drift, sigma=sigma_trend, shape=N)

    # Seasonality (The Cyclic Shock)
    b_sin = pm.Normal("b_sin", mu=0, sigma=10)
    b_cos = pm.Normal("b_cos", mu=0, sigma=10)
    season_component = pm.Deterministic("season_component", b_sin * np.sin(2*np.pi*t_months/12) + b_cos * np.cos(2*np.pi*t_months/12))

    # Likelihood
    sigma_obs = pm.HalfNormal("sigma_obs", sigma=5)
    mu = pm.Deterministic("mu", trend_component + season_component)
    y = pm.Normal("y", mu=mu, sigma=sigma_obs, observed=demand_total)

    prior_checks = pm.sample_prior_predictive(samples=500, random_seed=42)
    trace_bsts = pm.sample(draws=2000, tune=1000, cores=2, chains=2, target_accept=0.95, random_seed=42)
    post_checks = pm.sample_posterior_predictive(trace_bsts, random_seed=42)

    trace_bsts.extend(prior_checks)
    trace_bsts.extend(post_checks)

### PHASE 1: The Prior Predictive Check (The Sanity Test)
Before MCMC starts, we ask the math: *"If these Priors represent reality, what kind of data would they generate?"*
If the prior predictive check spits out values like -1,000,000 or +10 billion for "Demand," your physical constraints (Priors) are fatally wrong. The walker will get lost.

In [ ]:
# Plot: Priors vs Baseline Reality
fig, ax = plt.subplots(figsize=(10, 5))

# Plot the 500 "Blind" predictions our Math made BEFORE seeing the actual data
az.plot_ppc(trace_bsts, group="prior", data_pairs={"y": "y"}, color="blue", ax=ax, alpha=0.3)
ax.set_title("The Sanity Test: Do our Priors generate impossible physics?")
plt.show()

### PHASE 2: The MCMC Diagnostic Dashboard (Did the Detective get stuck?)
Check the math's convergence. 
*   **Trace (Left):** "Hairy caterpillars" = Healthy. "Snake" = The MCMC is stuck (divergent).
*   **Posterior (Right):** Smooth probability hills.
*   **R_hat:** Needs to be `< 1.01`. If it's `1.15`, the algorithm broke down; your results are lies.

In [ ]:
# We plot the fundamental structural parameters (skip 'trend_component' because it's 36 parameters)
az.plot_trace(trace_bsts, var_names=["sigma_trend", "sigma_obs", "b_sin", "b_cos"], compact=False)
plt.suptitle("MCMC Health", y=1.02)
plt.show()

# Check if max r_hat is < 1.01. If true, the chains converged mathematically.
summary_dfs = az.summary(trace_bsts, var_names=["sigma_trend", "sigma_obs", "b_sin", "b_cos"])
display(summary_dfs)

### PHASE 3: The Posterior Predictive Check (The Final Reality Test)
Did our mathematical model successfully learn the latent, hidden reality from the chaos data?
We ask the Posterior (having learned from the data) to simulate 1000 datasets and compare them exactly to the Black Dots (our real data). If the black line fits inside the blue cloud, we have mapped reality.

In [ ]:
# The Reality Test (Data vs Model's Best Understanding)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot the 2000 draws our Math generated AFTER learning from the Black Dots
az.plot_ppc(trace_bsts, num_pp_samples=100, data_pairs={"y": "y"}, color="blue", ax=ax)
ax.set_title("The Final Validation: Does our Model explain the true Fossil (Data)?")
plt.show()

In [ ]:
# Extract the mean of the Posterior to plot the "Best Guess" lines
post_trend = trace_bsts.posterior["trend_component"].mean(dim=["chain", "draw"])
post_season = trace_bsts.posterior["season_component"].mean(dim=["chain", "draw"])

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(t_months, demand_total, "k.", label="Raw Total Demand")
axes[0].plot(t_months, post_trend, "r-", label="Extracted Trend (Latent)")
axes[0].legend()

axes[1].plot(t_months, post_season, "b-", label="Extracted Cyclic Seasonality")
axes[1].legend()

plt.tight_layout()
plt.show()

### PHASE 4: The Residual Audit (What did we fail to explain?)
**Residual = Actual Data (The Fossil) - Model's Best Guess (Trend + Seasonality)**

**1. The Timeline (Manual Plot - Left Graph)**
*   **X-axis:** Time (Months).
*   **Y-axis:** How wrong the model was (Error Magnitude).
*   **What to look for:** A flat, boring horizontal cloud of white noise. The black dots should bounce randomly around the red Zero-line, staying mostly inside the red 2-Sigma (95% confidence) shading.
*   **Failure:** A diagonal drift or sudden, permanent spike outside the red bounds means your model's structural logic missed a massive physical shift in the data over time.

**2. The Geometry (ArviZ Built-in `plot_bpv` - Right Graph)**
*   **X-axis:** The Bayesian p-value (from 0 to 1). It ranks how extreme your actual data was compared to the model's simulated futures.
*   **Y-axis:** The density/frequency of those ranks.
*   **The Anatomy:**
    *   **The Horizontal White Line (y=1.0):** Built-in by ArviZ. It represents perfect, mathematically ideal uniform noise. Since the X-axis is 0 to 1, a perfect square uniform distribution has a height of exactly 1.0. 
    *   **The Grey Band:** Built-in by ArviZ. This is the allowed margin of error. It is extremely wide here (spanning ~0.5 to ~1.8) **purely because we only have 36 data points**. The algorithm is unsure. If we had 3,600 data points, the gray band would shrink to a razor-thin line exactly covering the white 1.0 line.
    *   **The Blue Curve:** The reality of your model's error distribution.
*   **Failure:** The blue curve *must* stay inside the grey band. If the blue curve blasts through the ceiling in a massive U-shape or tilts heavily to one side, your foundational assumption that errors follow `pm.Normal()` was illegal. The real-world errors are heavily skewed.

In [ ]:
# PHASE 4: The Residual Audit (The Architecture of Error)

# We must extract .values to strip xarray metadata.
# Otherwise, xarray tries to broadcast "trend_dim" and "season_dim" into a 36x36 matrix!
post_trend_vals = post_trend.values
post_season_vals = post_season.values

# Calculate the Residuals explicitly for the Timeline check
# Residual = Observed Fossil Data - Model's Best Guess
residuals = demand_total - (post_trend_vals + post_season_vals)

# Extract the mean of the posterior sigma to plot the error boundaries bounds
post_sigma_obs = trace_bsts.posterior["sigma_obs"].mean().item()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# The Timeline of Ignorance (Manual Plot)

axes[0].plot(t_months, residuals, "ko-", alpha=0.7, markersize=4)
axes[0].axhline(0, color="red", linestyle="--", linewidth=2)
axes[0].fill_between(t_months, -2*post_sigma_obs, 2*post_sigma_obs, color="red", alpha=0.1, label="Expected Error Bounds (2 Sigma)")
axes[0].set_title("The Timeline of Ignorance (Residuals over Time)")
axes[0].set_xlabel("Months")
axes[0].set_ylabel("Error Magnitude (Actual - Predicted)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# ArviZ Built-in Residual Geometry (Bayesian p-value)
az.plot_bpv(trace_bsts, kind="u_value", ax=axes[1])
axes[1].set_title("The Geometry of Error (ArviZ plot_bpv)")

plt.tight_layout()
plt.show()

In [ ]:
# DIAGNOSTIC: AUTOCORRELATION
# Why both plots?
# 1. Timeline plot (above) only proves MAGNITUDE: Do the errors stay inside the red box?
# 2. But human eyes can't see if errors are "holding hands" (e.g., today's error predicts tomorrow's error).
# This Autocorrelation plot does the math to prove the absence of MEMORY.

# HOW TO READ THIS:
# - EXPECTED (True White Noise): Blue spikes at Lag 1, Lag 2, etc. are tiny, hovering near the 0.0 line. At 0 large, corr with itself.
# - FAILURE (Leaky Physics): Large spikes at Lag > 0. It means your model missed a pattern.

fig, ax = plt.subplots(figsize=(10, 4))
ax.acorr(residuals, maxlags=15)
ax.set_title("Forensic Audit: Residual Autocorrelation")
ax.set_xlabel("Lags (Months)")
ax.set_ylabel("Correlation")
plt.show()

In [ ]:
# =====================================================================
# TOPIC: REPLACING GAUSSIAN RANDOM WALK WITH SIMPLE INTERCEPT/SLOPE
# =====================================================================

# WHY ITS IS NECESSARY:
# Our true physical baseline is a rigid, perfectly straight line (demand = 0.5 * t_months).
# A pm.GaussianRandomWalk without a defined 'drift' acts like a loose rope dropped on the floor.
# It structurally sags toward zero against the pull of a constantly increasing linear dataset.
# To model a system with constant linear gravity, we rip out the flexible chain and weld a rigid rod.


# remove
# trend_component = pm.GaussianRandomWalk("trend_component", sigma=sigma_trend, shape=N)

# add
# starting anchor of the straight line on the Y-Axis (Month 0 value)
# intercept = pm.Normal("intercept", mu=0, sigma=10)

# rate of change of the line per month
# slope = pm.Normal("slope", mu=0, sigma=2)

# trend_component = pm.Deterministic("trend_component", intercept + slope * t_months)

# this produeced a less wiggly (more straight bpv curve) and good priors and posterior grpahs signaling this is better
